### Relax Z homostatic condition to check robustness

In [ ]:
import os
import sys
import numpy as np

# Add the project root to the Python PATH
script_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(script_dir, '../../models/model_pet_nmda_vary_Z'))
sys.path.append(project_root)

from src.data import load_and_preprocess_data as get_data
from src.models import WorkingMemoryNetwork
from config.WM_params import WorkingMemoryParameters
import time
np.set_printoptions(threshold=np.inf)

random_seed = 666
outside_rng = np.random.default_rng(random_seed)

data = get_data(data_folder_path='../../data/processed', parcellation='schaefer200')
area_names = data['area_names']
SLN = data['SLN']
W = data['W']
h = data['hier_vals']
normalized_nmda_rd = data['nmda_rd']

# Initial condition and parameters
N = len(area_names)
Y0 = np.zeros((N, 8))
Y0[:, 0:3] = 5 * (1 + np.tanh(outside_rng.normal(0, 2, (N, 3)))) # r values are initialized randomly between 0 and 10
WM_params = WorkingMemoryParameters()
z_factor = 1 # Change this between 0.8 and 1.2 to see if the results still hold
WM_params.update(G=0.83, ALPHA=1.34, J_MAX=0.3) # Update G and alpha to optimal settings
# Generate the network
WM_network = WorkingMemoryNetwork(area_names=area_names, Y0=Y0, W=W, F=SLN, h=h,
                                  normalized_nmda_rd=normalized_nmda_rd,
                                  params=WM_params, gradient='NMDA', z_factor=z_factor)
# Run the simulation
t_end = 7
dt = 0.5e-3
stim_start_time = 2
stim_duration = 0.5
t_eval = np.arange(0, t_end+dt, dt) # Time points of the simulation
I_ext_strength = np.zeros((N, 3))

# Target names for which we want to assign 0.3
target_names = ['Vis_5', 'Vis_7', 'Vis_9']
# Find indexes of the target names in the list
rows = np.where(np.isin(area_names, target_names))[0]
I_ext_strength[rows, 0] = 0.3

start = time.perf_counter()
state_history, input_history = WM_network.run(t_end=t_end, dt=dt, I_ext_strengths=np.stack([I_ext_strength]),
                               ts_ext_start=np.array([stim_start_time]),
                               ts_ext_end=np.array([stim_start_time + stim_duration]))
end = time.perf_counter()
print("Elapsed (after compilation) = {}s".format((end - start)))

avg_firing_rates_A, avg_firing_rates_B = WM_network.save_avg_firing_rates_last_two_seconds(variables=['r_A', 'r_B'])
state_history_extract = WM_network.get_state_histories(['r_A'])[-1]
np.save(f'plots/vary_Z/cue_FR_Z_{z_factor}.npy', state_history_extract[int(stim_start_time / dt) + 100, :, 0])
avg_firing_rates = WM_network.save_avg_firing_rates_last_two_seconds()
np.save(f'plots/vary_Z/delay_FR_Z_{z_factor}.npy', avg_firing_rates)

_, pops = WM_network.check_sustained_activity_after_stim_removal()

# Need condition to check if it's A or B there, right now r_B is at index 0, it gets pops_A as pops_B.
try:
    pops_A = pops[0][1]
except:
    pops_A = []
try:
    pops_B = pops[1][1]
except:
    pops_B = []

print('Nodes where population B is persistent:', pops_B)
spont_switch, pop_switch = WM_network.check_spontaneous_switch_to_high_state(stim_start_time)
print(f'There is spontaneous switch to higher state: {spont_switch}, in populations {pop_switch}')
changes_detected, changed_populations = WM_network.check_sustained_activity_after_stim_removal()
print(changed_populations)
try:
    print('Number of persistent areas:', len(changed_populations[0][1]))
    areas_persistent = [area_names[i] for i in changed_populations[0][1]]
    print('Areas persistent:', areas_persistent)
    np.save(f'plots/vary_Z/persistent_areas_Z_{z_factor}.npy', areas_persistent)
except:
    print('No persistent areas')

# Plot the results
area_names_for_figure = ['Early Visual Cortex 1', 'Early Visual Cortex 2', 'Area p47r',
                         '9m', '6ma', 'PGi',
                         'Area 46', 'Area 9-46v']
_ = WM_network.plot_n_areas(xlim=(-2, t_end - 2), ylim=(-5, 65),
                            target_area_names=['Vis_9', 'Vis_5', 'Cont_PFCl_1', 'Default_PFC_7',
                                               'DorsAttn_FEF_2', 'Default_Par_4', 'SalVentAttn_PFCl_1',
                                               'Cont_PFCl_3'],
                            names_for_figures=area_names_for_figure,
                            variables=['r_A', 'r_B'], colors=['#56c68a', '#eaa8b7'],
                            save_path=f'plots/vary_Z/persistent_activity_Z_{z_factor}.svg',
                            legend=False,
                            show=False)
_ = WM_network.plot_avg_firing_rates_last_two_seconds(save_path=f'plots/vary_Z/avg_last_2_s_Z_{z_factor}.svg',
                                                       color=['#56c68a', '#eaa8b7'], show=False)

/home/rad216/PhD_Projects/WM_human_merged/models/model_pet_nmda_vary_Z/src/models.py:432: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 2, 'C', False, aligned=True), Array(float64, 1, 'A', False, aligned=True))
  dY, I_mon = calc_dY(A=A, B=B, D=D,


Elapsed (after compilation) = 8.673861252027564s
Nodes where population B is persistent: []
There is spontaneous switch to higher state: False, in populations []
[('r_A', [31, 38, 39, 41, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 63, 64, 65, 66, 67, 72, 73, 74, 75, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 99])]
Number of persistent areas: 47
Areas persistent: ['DorsAttn_Post_2', 'DorsAttn_Post_9', 'DorsAttn_Post_10', 'DorsAttn_FEF_2', 'SalVentAttn_ParOper_2', 'SalVentAttn_ParOper_3', 'SalVentAttn_FrOperIns_1', 'SalVentAttn_FrOperIns_2', 'SalVentAttn_FrOperIns_3', 'SalVentAttn_FrOperIns_4', 'SalVentAttn_PFCl_1', 'SalVentAttn_Med_1', 'SalVentAttn_Med_2', 'SalVentAttn_Med_3', 'Limbic_OFC_1', 'Limbic_OFC_2', 'Limbic_TempPole_1', 'Limbic_TempPole_2', 'Limbic_TempPole_3', 'Limbic_TempPole_4', 'Cont_Par_1', 'Cont_Temp_1', 'Cont_OFC_1', 'Cont_PFCl_1', 'Cont_PFCl_2', 'Cont_PFCl_3', 'Cont_Cing_2', 'Default_Temp_1', 'Default_Temp_2', 'Default_Temp_

In [2]:
# Script to plot brain maps during the cue and delay period under optimal parameters
# for model built with nmda PET imaging
import numpy as np
import matplotlib.pyplot as plt
from nibabel.freesurfer.io import read_annot
from nilearn import datasets, plotting
import nibabel.freesurfer as fs
import scienceplots
plt.style.use(['science', 'nature'])
from matplotlib.colors import LinearSegmentedColormap
plot_params = {
    "figure.dpi": "600",
    "axes.labelsize": 20,
    "axes.linewidth": 1.5,
    "axes.titlesize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.title_fontsize": 16,
    "legend.fontsize": 13,
    "xtick.major.size": 3.5,
    "xtick.major.width": 1.5,
    "xtick.minor.size": 2.5,
    "xtick.minor.width": 1.5,
    "ytick.major.size": 3.5,
    "ytick.major.width": 1.5,
    "ytick.minor.size": 2.5,
    "ytick.minor.width": 1.5,
}
plt.rcParams.update(plot_params)
cmaps = {
    "excitation_seq": LinearSegmentedColormap.from_list(
        "exc_seq", ["#e7f6ed", "#86c3a2", "#56c68a"], N=256
    )
}
cue_FR = f'plots/vary_Z/cue_FR_Z_{z_factor}.npy' # Path to npy file containing the FR of brain areas at cue time (before persistent activation)
cue_FR = np.load(cue_FR)

labels_schaefer, ctab_schaefer, region_names_schaefer = fs.read_annot('../../data/raw/brain_annotations/Schaefer/fsaverage5/atl-Schaefer2018_space-fsaverage5_hemi-L_desc-200Parcels7Networks_deterministic.annot')
region_names_schaefer = [r.decode('utf-8') for r in region_names_schaefer]
mapped_data_schaefer_cue = np.zeros_like(labels_schaefer, dtype=float) # Define a map with the same number of vertices as the annotation file
for i, region in enumerate(region_names_schaefer):
    region_idx = i
    if i == 0:
        mapped_data_schaefer_cue[labels_schaefer == region_idx] = None
    else:
        mapped_data_schaefer_cue[labels_schaefer == region_idx] = cue_FR[region_idx - 1] # First index in region_names is background
vmin, vmax = np.nanmin(mapped_data_schaefer_cue), np.nanmax(mapped_data_schaefer_cue)

fs = datasets.fetch_surf_fsaverage('fsaverage5')
for view in ['lateral','medial']:
    fig = plt.figure(constrained_layout=True, figsize=(6,6))
    ax = fig.add_subplot(1,1,1, projection='3d')
    plotting.plot_surf_stat_map(
        fs.infl_left, mapped_data_schaefer_cue, fs.sulc_left,
        hemi='left', view=view,
        cmap=cmaps['excitation_seq'], colorbar=True,
        symmetric_cbar=False,
        vmin=0, vmax=vmax,
        axes=ax, figure=fig,
        engine='matplotlib', avg_method='median'
    )
    for ext in ['pdf']:
        dpi = 150 if ext in ['png'] else None
        fig.savefig(f'plots/vary_Z/cue_activity_{view}_{z_factor}.{ext}', bbox_inches='tight')
    plt.close()
    
path_avg_FR_persistent = f'plots/vary_Z/delay_FR_Z_{z_factor}.npy' # Path to npy file containing the average FR over last 2s of simulation
avg_FR_persistent = np.load(path_avg_FR_persistent)
mapped_data_schaefer_persistent = np.zeros_like(labels_schaefer, dtype=float) # Define a map with the same number of vertices as the annotation file
for i, region in enumerate(region_names_schaefer):
    region_idx = i
    if i == 0:
        mapped_data_schaefer_persistent[labels_schaefer == region_idx] = None
    else:
        mapped_data_schaefer_persistent[labels_schaefer == region_idx] = avg_FR_persistent[region_idx - 1] # 
fsaverage = datasets.fetch_surf_fsaverage('fsaverage5') # Extract surface to plot on
vmin, vmax = np.nanmin(mapped_data_schaefer_persistent), np.nanmax(mapped_data_schaefer_persistent)

fs = datasets.fetch_surf_fsaverage('fsaverage5')
for view in ['lateral','medial']:
    fig = plt.figure(constrained_layout=True, figsize=(6,6))
    ax = fig.add_subplot(1,1,1, projection='3d')
    plotting.plot_surf_stat_map(
        fs.infl_left, mapped_data_schaefer_persistent, fs.sulc_left,
        hemi='left', view=view,
        cmap=cmaps['excitation_seq'], colorbar=True,
        symmetric_cbar=False,
        vmin=0, vmax=vmax,
        axes=ax, figure=fig,
        engine='matplotlib', avg_method='median'
    )
    for ext in ['pdf']:
        dpi = 150 if ext in ['png'] else None
        fig.savefig(f'plots/vary_Z/persistent_activity_{view}_{z_factor}.{ext}', bbox_inches='tight')
    plt.close()